In [ ]:
import os
import sys
if 'google.colab' in sys.modules:
  !pip install -q numba imageio
  !apt-get install -y -qq ffmpeg

os.makedirs('../results/gifs', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import PillowWriter, FuncAnimation
from PIL import Image, ImageSequence

In [ ]:
colors = [['dodgerblue', 'indigo'],
          ['orangered', 'sienna'],
          ['gold', 'olive']]

n = 3930
G, M, m, dt = 0.01, 500, 0.1, 0.001
r, p, F = [np.zeros((n, 2)) for _ in range(3)]
r0, p0 = [2, 0], [0, 0.1]

In [ ]:
# Two body problem
def Euler(r0=r0, p0=p0):
  r[0], p[0] = r0, p0
  for i in range(n - 1):
    F[i] = - G * M * m / np.linalg.norm(r[i])**3 * r[i]
    r[i + 1] = r[i] + p[i] / m * dt + F[i] / (2 * m) * dt**2
    p[i + 1] = p[i] + F[i] * dt
  return [r, p]

def Verlet(r0=r0, p0=p0):
  r[0], p[0] = r0, p0
  r[-1] = r[0] - p[0] / m * dt
  for i in range(n - 1):
    F[i] = - G * M * m / np.linalg.norm(r[i])**3 * r[i]
    r[i + 1] = 2 * r[i] - r[i - 1] + F[i] / m * dt**2
    p[i] = (r[i + 1] - r[i]) * m / dt 
  return [r, p]

def Zabka(r0=r0, p0=p0):
  r[0], p[0] = r0, p0
  for i in range(n - 1):
    F[i] = - G * M * m / np.linalg.norm(r[i])**3 * r[i]
    p[i + 1] = p[i] + F[i] * dt
    r[i + 1] = r[i] + p[i + 1] / m * dt
  return [r, p]

In [ ]:
def plot_orbit(method):
  if method == 'euler':
    points = Euler()
    name = 'Eulera'
  elif method == 'verlet':
    points = Verlet()
    name = 'Verleta'
  elif method == 'zabka':
    points = Zabka()
    name = 'Zabki'
  else:
    raise ValueError(f"Unknown method: {method}")
  
  fig, ax = plt.subplots()
  fig.suptitle(f'Orbita dla {name}')
  ax.scatter(0, 0, color=colors[1][0])
  ax.scatter(points[0][:,0], points[0][:,1], s=.1, color=colors[0][0])
  fig.tight_layout()
  plt.show()
  plt.close(fig)

  K, P, E = [np.zeros((n, 1)) for _ in range(3)]  
  t = np.array(range(n)) * dt
  for i in range(n):
    P[i] = -G * M * m / np.linalg.norm(points[0][i])
    K[i] = np.linalg.norm(points[1][i])**2 / (2 * m)
    E[i] = K[i] + P[i]

  fig, ax = plt.subplots()
  fig.suptitle(f'Energie od czasu dla {name}')
  ax.set_xlabel("$t$")
  ax.plot(t[:-1], P[:-1], color=colors[0][0], label='Potential')
  ax.plot(t[:-1], K[:-1], color=colors[1][0], label='Kinetic')
  ax.plot(t[:-1], E[:-1], color=colors[2][0], label='Energy')
  ax.legend()
  fig.tight_layout()
  plt.show()
  plt.close(fig)

plot_orbit('verlet')

In [ ]:
points = Verlet()

fig, ax = plt.subplots()
ax.set_xlim(-1, 2.5)
ax.set_ylim(-1.5, 1.5)
ax.axis('off')
ax.scatter(0, 0, c=colors[2][0], s=70)

scatter = ax.scatter([], [], c=colors[2][1], s=50)

def update(frame):
  x, y = points[0][frame*30]
  scatter.set_offsets([[x, y]])
  return [scatter]

gif_name = "../results/gifs/04_verlet_figure.gif"
anim = FuncAnimation(fig, update, frames=int(n/30), blit=True, repeat=True)
anim.save(gif_name, writer=PillowWriter(fps=30))

im = Image.open(gif_name)
frames = [frame.copy() for frame in ImageSequence.Iterator(im)]

frames[0].save(
  "temp.gif",
  save_all=True,
  append_images=frames[1:],
  loop=0,
  duration=2,
)

In [ ]:
# Three body problem
def Chencinera():
  G = m1 = m2 = m3 = 1
  F1, F2, F3, r1, r2, r3, p1, p2, p3 = [np.zeros((n, 2)) for _ in range(9)]
  r0, v0 = np.array([0.97000436, -0.24308753]), np.array([0.93240737, 0.86473146])
  r1[0], r2[0], r3[0] = r0, (-1) * r0, [0, 0]
  p1[0], p2[0], p3[0] = v0 * m1 / (-2), v0 * m2 / (-2), v0 * m3

  r1[-1] = r1[0] - p1[0] / m1 * dt
  r2[-1] = r2[0] - p2[0] / m2 * dt
  r3[-1] = r3[0] - p3[0] / m3 * dt

  for i in range(n-2):
    r12 = r1[i] - r2[i]
    r13 = r1[i] - r3[i]
    r23 = r2[i] - r3[i]

    F1[i] = -G * m1 * (m2 * r12 / np.linalg.norm(r12)**3 + m3 * r13 / np.linalg.norm(r13)**3)
    F2[i] = -G * m2 * (m1 * (-r12) / np.linalg.norm(r12)**3 + m3 * r23 / np.linalg.norm(r23)**3)
    F3[i] = -G * m3 * (m1 * (-r13) / np.linalg.norm(r13)**3 + m2 * (-r23) / np.linalg.norm(r23)**3)

    r1[i + 1] = 2 * r1[i] - r1[i - 1] + F1[i] / m1 * dt**2
    r2[i + 1] = 2 * r2[i] - r2[i - 1] + F2[i] / m2 * dt**2
    r3[i + 1] = 2 * r3[i] - r3[i - 1] + F3[i] / m3 * dt**2

    p1[i] = (r1[i + 1] - r1[i]) * m1 / dt 
    p2[i] = (r2[i + 1] - r2[i]) * m2 / dt 
    p3[i] = (r3[i + 1] - r3[i]) * m3 / dt 

  return [r1, r2, r3]

In [ ]:
n = 2000
bodies = Chencinera()

fig, ax = plt.subplots()
fig.suptitle(f'Three body problem')
ax.scatter(0, 0, color=colors[1][0])
for i, label in enumerate(['body1', 'body2', 'body3']):
  ax.scatter(bodies[i][:, 0], bodies[i][:, 1], color=colors[i][0], label=label)

fig.tight_layout()
ax.legend()
plt.show()
plt.close(fig)


In [ ]:
n = 15800
bodies = Chencinera()

fig, ax = plt.subplots(figsize=(5,5))
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.axis('off')

scatters = [ax.scatter([], [], s=100, c=colors[i][0]) for i in range(3)]

def update(frame):
  step = frame * 20
  for i, r in enumerate(bodies[:3]):
    x, y = r[step]
    scatters[i].set_offsets([[x, y]])
  return scatters

gif_name = "../results/gifs/04_chenciner_figure.gif"
anim = FuncAnimation(fig, update, frames=int(n/50), blit=True, repeat=True)
anim.save(gif_name, writer=PillowWriter(fps=30))
plt.close(fig)

im = Image.open(gif_name)
frames = [frame.copy() for frame in ImageSequence.Iterator(im)]

frames[0].save(
  "temp.gif",
  save_all=True,
  append_images=frames[1:],
  loop=0,
  duration=2,
)